# 🌐 Notebook 03: APIs & Web Scraping

**Szenario**: Deine Aufgabe ist es, Wetterdaten über eine öffentliche API abzufragen und Wikipedia-Tabellen zu scrapen.

**Lernziele:**
- REST-APIs mit requests abfragen
- JSON-Responses in DataFrames umwandeln
- Einfaches Web Scraping mit BeautifulSoup
- Ethische Grundsätze beim Scraping

---

## 1️⃣ REST-APIs – JSONPlaceholder (öffentlich, kein Key)

In [ ]:
import requests
import pandas as pd

# Öffentliche Test-API (kein API-Key nötig)
url = 'https://jsonplaceholder.typicode.com/posts'
response = requests.get(url, timeout=10)

print(f'Status Code: {response.status_code}')
print(f'Content-Type: {response.headers.get("content-type", "?")}')  
data = response.json()
print(f'Anzahl Einträge: {len(data)}')
print(f'Erster Eintrag: {data[0]}')

# In DataFrame laden
df = pd.DataFrame(data)
print(f'\nDataFrame: {df.shape}')
print(df.head(3))

In [ ]:
# HTTP Status Codes verstehen
status_codes = {
    200: '✅ OK – Erfolg',
    201: '✅ Created – Ressource erstellt',
    400: '❌ Bad Request – Fehlerhafte Anfrage',
    401: '❌ Unauthorized – Nicht autorisiert',
    403: '❌ Forbidden – Verboten',
    404: '❌ Not Found – Nicht gefunden',
    429: '⚠️ Too Many Requests – Rate Limit',
    500: '❌ Internal Server Error – Serverfehler',
}

print('=== HTTP Status Codes ===')
for code, desc in status_codes.items():
    print(f'  {code}: {desc}')

In [ ]:
# API mit Parametern – User-Posts
url = 'https://jsonplaceholder.typicode.com/posts'
params = {'userId': 1}
response = requests.get(url, params=params, timeout=10)
user_posts = pd.DataFrame(response.json())
print(f'Posts von User 1: {len(user_posts)}')
print(user_posts[['id','userId','title']].head())

# Daten in SQLite speichern
import sqlite3
conn = sqlite3.connect('../data/dav_sample.db')
df.to_sql('api_posts', conn, if_exists='replace', index=False)
conn.close()
print('\n✅ Daten in SQLite gespeichert!')

## 2️⃣ Web Scraping mit BeautifulSoup

⚠️ **Ethische Grundsätze:**
- robots.txt immer prüfen
- Nutzungsbedingungen lesen  
- Rate Limiting (time.sleep zwischen Anfragen)
- Keine personenbezogenen Daten ohne Einwilligung

In [ ]:
from bs4 import BeautifulSoup
import time

# Wikipedia-Tabelle scrapen (öffentlich, erlaubt)
url = 'https://en.wikipedia.org/wiki/List_of_largest_airlines_in_Europe'
headers = {'User-Agent': 'DAV-Lab-Student/1.0 (educational; course project)'}

try:
    response = requests.get(url, headers=headers, timeout=15)
    print(f'Status: {response.status_code}')
    
    soup = BeautifulSoup(response.text, 'html.parser')
    
    # Alle Tabellen finden
    tables = soup.find_all('table', class_='wikitable')
    print(f'Wikitabellen gefunden: {len(tables)}')
    
    if tables:
        # Erste Tabelle parsen
        rows = tables[0].find_all('tr')
        print(f'Zeilen in erster Tabelle: {len(rows)}')
        # Kopfzeile
        headers_row = [th.text.strip() for th in rows[0].find_all(['th','td'])]
        print(f'Spalten: {headers_row[:5]}')
except Exception as e:
    print(f'⚠️ Scraping nicht möglich (kein Internet?): {e}')
    print('Das ist OK – Web Scraping benötigt eine Internetverbindung.')

In [ ]:
# Einfacherer Weg: pd.read_html()
try:
    dfs = pd.read_html('https://en.wikipedia.org/wiki/List_of_largest_airlines_in_Europe')
    print(f'Tabellen gefunden: {len(dfs)}')
    if dfs:
        print(dfs[0].head(5))
except Exception as e:
    print(f'⚠️ Nicht möglich: {e}')

## ✅ Challenge

1. Frage die URL `https://jsonplaceholder.typicode.com/users` ab und lade alle User in einen DataFrame
2. Extrahiere nur: id, name, email, city (aus address.city)
3. Speichere das Ergebnis in SQLite als Tabelle 'api_users'

💡 Tipp: `pd.json_normalize()` hilft bei verschachtelten JSON-Objekten

In [ ]:
# Deine Lösung:



---
**Weiter mit:** `04_Daten_Inspizieren.ipynb`